# TabICLv2 Classifier — DIMER E2E tabular classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabicl-classifier-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabicl-classifier-pipeline/blob/main/tutorials/tabiclv2_classifier_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-jingang%2FTabICL-ffcc4d?style=flat)](https://huggingface.co/jingang/TabICL) [![Upstream](https://img.shields.io/badge/Upstream-soda--inria%2Ftabicl-181717?style=flat&logo=github&logoColor=white)](https://github.com/soda-inria/tabicl) [![arXiv](https://img.shields.io/badge/arXiv-2602.11139-b31b1b.svg)](https://arxiv.org/abs/2602.11139)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** end-to-end TabICLv2 tabular classification: immutable checkpoint acquisition, validated support data with class coverage, in-context evaluation against trivial and classical baselines, optional gradient fine-tuning, new-data inference with class probabilities, a DIMER-style serving bundle and its fresh reload

**This notebook is standalone.** It carries the repository's pipeline module (`src/tabicl_classifier_pipeline/api.py` at revision `dfc05123e8c7`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `4dcd344ece2c00be9e831fdd35bed57b5ad83e19` (~110 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned TabICLv2 checkpoint, loads scikit-learn's bundled Breast Cancer Wisconsin table (no download), validates the tables into an input manifest and encodes them, evaluates the pretrained TabICLv2 classifier **adapted by in-context conditioning on the training split** (the adaptation stage that runs by default — no gradient update), fits classical tree baselines and a majority-class baseline for comparison and writes the evaluation report, exports a DIMER-style serving bundle and reloads it from disk to prove the fresh boundary. Gradient fine-tuning (`FinetunedTabICLClassifier`) is an optional experiment (`RUN_FINE_TUNING`, off by default, Section 6) because it needs a GPU-sized time budget; a reviewer reading NOTEBOOK_SPEC 2.0 RUN7/FT2 as requiring gradient adaptation on the default path should treat that as an open decision. No repository clone, DIMER worker or service, credential, upload dialog or configuration edit is required (§5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one labelled CSV (or pre-split files); it enters the same validation, encoding, in-context conditioning, baseline, evaluation, export and fresh-reload cells as the sample (DAT14), and `RUN_NEW_DATA_INFERENCE` in Section 8 scores your own unlabelled rows with class probabilities. Expected schema, ceilings and privacy guidance are stated in the Prerequisites and in Section 4; uploads stay inside this runtime. BYOD is optional and never part of the default path.

TabICLv2 is an in-context tabular foundation model: `fit` registers the (encoded) training rows as the model's context and every `predict_proba` call feeds context plus query rows through the transformer and reads off a class distribution per query row — no gradient step happens unless you opt into the fine-tuning gate. The upstream project supplies the model and the checkpoint; the carried package adds the pinned snapshot scheme, the table preparation, class-coverage and encoding rules, the metric set, the `validate_inputs` / `majority_class_baseline` / `evaluation_report` helpers, and the serving-bundle safety checks. `predict` applies an implicit `argmax` over class probabilities that are raw ensemble outputs, **not calibrated probabilities**; the package ships no acceptance threshold. The default sample is scikit-learn's bundled Breast Cancer Wisconsin table; its metrics are tutorial sanity evidence, not a benchmark or production claim.

**Learning objectives:** install the pinned runtime, read what the carried package guarantees, resolve and digest-verify the immutable upstream checkpoint, load a public sample (binary, three-class, or mixed categorical/numeric) or your own CSV(s) and validate them into an input manifest with class coverage preserved, evaluate the pretrained model on a holdout and an independent test partition against the majority-class baseline, optionally fine-tune on CUDA with holdout-based selection, benchmark LightGBM and Random Forest on the same partitions with strict label alignment and blend in memory, write an evaluation report, optionally score new rows with class probabilities under an explicit `argmax` rule, export a DIMER-style serving bundle and prove it reloads from a fresh directory.

**This notebook does not demonstrate:** regression, forecasting, calibrated probabilities, or any deployment threshold. `prediction` is the `argmax` over uncalibrated class probabilities; the fine-tuning path runs only on CUDA and only when `RUN_FINE_TUNING` is switched on.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.11+; the pins were resolved on Python 3.12). The default path runs on CPU and uses CUDA automatically when available; the fine-tuning gate requires CUDA. The pinned `torch==2.14.0` install is the largest download of the run.
- **Knowledge:** basic pandas; what a stratified holdout, an independent test partition, accuracy, balanced accuracy, log loss and ROC-AUC are.
- **Data:** the default sample is scikit-learn's bundled Breast Cancer Wisconsin table (569 rows, 30 numeric features, two classes), loaded from the installed package, so nothing is downloaded and no private data is needed; `Sample: Wine` is the bundled three-class table and `Sample: Palmer Penguins` fetches the repository's pinned CC0 archive (mixed categorical/numeric features, pre-split) by immutable commit and verifies its SHA-256. BYOD upload (one CSV with a `target` column, or pre-split `train.csv`/`val.csv`/`test.csv`) is selected through `DATA_SOURCE` and is off by default so the sample path runs top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `jingang/TabICL` snapshot (~110 MB in total) at revision `4dcd344ece2c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy`, `pandas`, `sklearn` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'tabicl==2.1.1',
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'numpy==2.5.2',
    'pandas==3.0.5',
    'scikit-learn==1.9.0',
    'pyarrow==25.0.1',
    'lightgbm==4.7.0',
    'huggingface-hub==1.30.0',
    'transformers==5.17.0',
    'wandb==0.30.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'tabicl-classifier-pipeline',
    'repository_revision': 'dfc05123e8c7b2d063041e946801ec3fd227158b',
    'embedded_module': 'src/tabicl_classifier_pipeline/api.py',
    'embedded_modules': ['src/tabicl_classifier_pipeline/api.py'],
    'module_sha256': 'ea063ec901b144df26b46304bf33f73a8b33555340560c365ebce986f193fbd0',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy, pandas, sklearn
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/tabicl_classifier_pipeline/` @ `dfc05123e8c7`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/tabicl_classifier_pipeline/api.py`

In [ ]:
# ruff: noqa: E501  -- long docstrings, messages and single-line test fixtures are kept readable
from __future__ import annotations

import csv
import hashlib
import importlib.metadata
import io
import json
import math
import os
import shutil
import stat
import sys
import zipfile
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

ARTIFACT_FORMAT = "tabicl-dimer-classifier-v1"

# Fleet snapshot identity (DIMER Notebook Specification 1.1, ST3/MOD13). The pinned upstream checkpoint is
# unchanged; these are the fleet-standard names for the same repository, revision, license and snapshot key.
# The BASE_* spellings below stay as the package's published names and alias these constants.
MODEL_ID = "jingang/TabICL"
MODEL_REVISION = "4dcd344ece2c00be9e831fdd35bed57b5ad83e19"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "tabicl-classifier-v2"
MANIFEST_NAME = "dimer-base-manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative

BASE_MODEL_REPO = MODEL_ID
BASE_CHECKPOINT_NAME = "tabicl-classifier-v2-20260212.ckpt"
BASE_MODEL_REVISION = MODEL_REVISION
BASE_MODEL_SHA256 = "bdc7dbd5e4ff21f8f0456fcf90c6b7cdf72dbea960f2d05b19bec19f9b3d4ed0"

# Operational ceilings the tutorials enforce before any model execution (DAT22).
MIN_TRAIN_ROWS = 50  # labelled support rows required to condition the classifier
MIN_EVAL_ROWS = 2  # labelled rows required for a holdout / test partition
MAX_TRAIN_ROWS = 50_000  # support rows per conditioning call
MAX_FEATURES = 2_000  # feature columns per table
MAX_ARTIFACT_EXPANDED_BYTES = 1024 * 1024 * 1024  # 1 GiB expanded ZIP size for a serving artifact
DEFAULT_N_ESTIMATORS = 8
DEFAULT_RANDOM_STATE = 42
MIN_CLASSES = 2  # distinct target classes in the support rows
MIN_ROWS_PER_CLASS = 2  # rows per class for a stratified holdout split
DECISION_RULE = "argmax"  # `predict` is the argmax over `predict_proba`; no threshold is shipped
METRIC_IDS = ("accuracy", "balanced_accuracy", "f1_weighted", "log_loss", "roc_auc")  # the ids `classification_metrics` reports


def runtime_identity() -> dict[str, str]:
    import torch

    return {
        "pythonVersion": sys.version.split()[0],
        "tabiclVersion": importlib.metadata.version("tabicl"),
        "torchVersion": torch.__version__,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }


def create_classifier(
    *,
    model_path: str | Path,
    n_estimators: int,
    random_state: int,
    device: str,
    allow_auto_download: bool = False,
    support_many_classes: bool = True,
):
    """Construct the supported TabICLv2 classification serving estimator."""
    from tabicl import TabICLClassifier

    return TabICLClassifier(
        model_path=str(model_path),
        allow_auto_download=allow_auto_download,
        n_estimators=n_estimators,
        random_state=random_state,
        device=device,
        support_many_classes=support_many_classes,
    )


def create_finetuned_classifier(**kwargs: Any):
    """Construct the supported TabICLv2 classification fine-tuning estimator."""
    from tabicl import FinetunedTabICLClassifier

    kwargs.setdefault("allow_auto_download", False)
    kwargs.setdefault("support_many_classes", True)
    return FinetunedTabICLClassifier(**kwargs)


def fine_tune_classifier(model: Any, X: Any, y: Any, **kwargs: Any) -> Any:
    """Run the upstream fine-tuning operation through the repository API."""
    return model.fit(X, y, **kwargs)


def condition_classifier(model: Any, X: Any, y: Any) -> Any:
    """Register the serving support context required by TabICL inference (no gradient step)."""
    model.fit(X, y)
    return model


def predict_labels(model: Any, X: Any) -> np.ndarray:
    """Return one-dimensional class labels (the argmax of the class probabilities)."""
    values = np.asarray(model.predict(X))
    if values.ndim != 1:
        values = values.reshape(-1)
    return values


def predict_probabilities(model: Any, X: Any) -> np.ndarray:
    """Return finite class probabilities, one column per entry of ``model.classes_`` in that order."""
    values = np.asarray(model.predict_proba(X), dtype=float)
    if values.ndim != 2 or values.shape[1] != len(model.classes_):
        raise RuntimeError("TabICL produced probabilities of an unexpected shape")
    if not np.isfinite(values).all():
        raise RuntimeError("TabICL produced non-finite class probabilities")
    return values


def align_probabilities(proba: Any, model_classes: Sequence[Any], classes: Sequence[Any]) -> np.ndarray:
    """Reorder probability columns from ``model_classes`` order into ``classes`` order (strict: every class in
    ``classes`` must be a column, so blending two models can never permute labels silently)."""
    proba = np.asarray(proba, dtype=float)
    lookup = {label: index for index, label in enumerate(model_classes)}
    missing = [label for label in classes if label not in lookup]
    if missing:
        raise RuntimeError(f"model never saw classes: {missing}")
    return proba[:, [lookup[label] for label in classes]]


def read_single_input(*, env_var: str, label: str) -> tuple[str, bytes]:
    """Read one user-supplied file from an explicit path or Colab upload dialog."""
    explicit = os.environ.get(env_var, "").strip()
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"{label} path does not exist: {path}")
        return path.name, path.read_bytes()

    try:
        from google.colab import files  # type: ignore
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            f"{label} requires either a Colab upload or environment variable {env_var}"
        ) from exc

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Upload exactly one {label}")
    name, payload = next(iter(uploaded.items()))
    return str(name), bytes(payload)


def download_output(path: str | Path) -> Path:
    """Download in Colab; otherwise retain the file at its explicit Jupyter path."""
    resolved = Path(path).resolve()
    try:
        from google.colab import files  # type: ignore
    except ModuleNotFoundError:
        print(f"Output retained at {resolved}")
        return resolved
    files.download(str(resolved))
    return resolved


def _base_version(value: str) -> str:
    return value.split("+")[0]


def validate_artifact_runtime(
    manifest: dict[str, Any],
    *,
    expected_artifact_format: str = ARTIFACT_FORMAT,
    expected_tabicl_version: str = "2.1.1",
    expected_torch_version: str = "2.14.0",
) -> dict[str, Any]:
    """Validate and return artifact/runtime provenance before model deserialization."""
    if manifest.get("artifactFormat") != expected_artifact_format:
        raise ValueError(f"Unsupported artifactFormat: {manifest.get('artifactFormat')}")
    if manifest.get("tabiclVersion") != expected_tabicl_version:
        raise ValueError("Artifact TabICL version does not match the supported runtime")

    observed = runtime_identity()
    if _base_version(observed["torchVersion"]) != expected_torch_version:
        raise RuntimeError(
            f"Runtime torch {observed['torchVersion']} is incompatible with the "
            f"release-verified torch {expected_torch_version}"
        )

    producer = manifest.get("runtime")
    compatibility = {
        "artifactFormat": manifest.get("artifactFormat"),
        "baseCheckpoint": manifest.get("baseCheckpoint"),
        "baseModelRevision": manifest.get("baseModelRevision"),
        "baseModelSha256": manifest.get("baseModelSha256"),
        "tabiclVersion": manifest.get("tabiclVersion"),
        "producerRuntime": producer,
        "consumerRuntime": observed,
        "policy": "TabICL exact; consumer torch base version 2.14.0; device may differ",
    }

    if producer:
        producer_torch = str(producer.get("torchVersion", ""))
        if producer_torch and _base_version(producer_torch) != expected_torch_version:
            raise RuntimeError(
                f"Artifact was produced with torch {producer_torch}; expected release family "
                f"{expected_torch_version}"
            )
    else:
        compatibility["warning"] = (
            "Legacy artifact has no runtime block; TabICL/version and digest checks apply, "
            "but producer runtime compatibility cannot be proven."
        )
    return compatibility


# ---------------------------------------------------------------------------
# Fleet snapshot scheme (NOTEBOOK_SPEC 1.1 ST3/ST4, MOD13): manifest-driven verification and staging.
# ---------------------------------------------------------------------------


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local pinned snapshot against its manifest; raise naming the first mismatch.

    The manifest is the parity anchor the standalone tutorials carry inline (ST3). The package's own
    ``BASE_MODEL_SHA256`` is not replaced by it: the manifest entry for ``BASE_CHECKPOINT_NAME`` must equal
    that constant, so the two can never diverge silently.
    """
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    entries = manifest.get("files", [])
    declared = {entry["path"]: entry["sha256"] for entry in entries}
    if declared.get(BASE_CHECKPOINT_NAME) != BASE_MODEL_SHA256:
        raise ValueError(
            f"manifest {BASE_CHECKPOINT_NAME} sha256 {declared.get(BASE_CHECKPOINT_NAME)!r} "
            f"!= BASE_MODEL_SHA256 {BASE_MODEL_SHA256!r}"
        )
    for entry in entries:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(repo_id=MODEL_ID, filename=relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; ``verify_snapshot`` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


class TabICLClassificationPipeline:
    """Serving wrapper: the digest-verified pinned checkpoint behind the upstream ``TabICLClassifier``.

    ``from_pretrained`` stages and verifies the snapshot and constructs the estimator through
    ``create_classifier`` (no auto-download); ``fit`` registers the support context (in-context learning,
    no gradient training), ``predict_proba`` returns finite class probabilities in ``classes_`` order and
    ``predict`` the ``argmax`` label (``DECISION_RULE``; no threshold is shipped).
    """

    def __init__(
        self,
        estimator: Any,
        *,
        model_path: Path,
        n_estimators: int,
        random_state: int,
        device: str,
        source: str = "local-snapshot",
    ) -> None:
        self.estimator = estimator
        self.model_path = Path(model_path)
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.device = device
        self.source = source
        self.is_fitted = False

    @classmethod
    def from_pretrained(
        cls,
        weights_dir: str | Path | None = None,
        *,
        allow_download: bool = False,
        n_estimators: int = DEFAULT_N_ESTIMATORS,
        random_state: int = DEFAULT_RANDOM_STATE,
        device: str | None = None,
    ) -> TabICLClassificationPipeline:
        """Stage what is missing (at ``MODEL_REVISION``), re-hash every manifest entry, then build the
        estimator on the verified checkpoint. ``tabicl`` deserialises the checkpoint on the first ``fit``."""
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        if device is None:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        model_path = root / BASE_CHECKPOINT_NAME
        estimator = create_classifier(
            model_path=model_path,
            n_estimators=n_estimators,
            random_state=random_state,
            device=device,
            allow_auto_download=False,
        )
        return cls(
            estimator,
            model_path=model_path,
            n_estimators=n_estimators,
            random_state=random_state,
            device=device,
        )

    @property
    def classes_(self) -> list[Any]:
        if not self.is_fitted:
            raise RuntimeError("Pipeline is not conditioned; call fit(X, y) with the support rows first")
        return list(self.estimator.classes_)

    def fit(self, X: Any, y: Any) -> TabICLClassificationPipeline:
        condition_classifier(self.estimator, X, y)
        self.is_fitted = True
        return self

    def predict_proba(self, X: Any) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("Pipeline is not conditioned; call fit(X, y) with the support rows first")
        return predict_probabilities(self.estimator, X)

    def predict(self, X: Any) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("Pipeline is not conditioned; call fit(X, y) with the support rows first")
        return predict_labels(self.estimator, X)


# ---------------------------------------------------------------------------
# Table preparation, encoding, metrics (extracted from the tutorials so both notebooks share one code path).
# ---------------------------------------------------------------------------


def _check_classification_table(
    frame: pd.DataFrame,
    target_column: str,
    *,
    min_rows: int,
    max_rows: int = MAX_TRAIN_ROWS,
    max_features: int = MAX_FEATURES,
    min_classes: int = MIN_CLASSES,
) -> tuple[pd.DataFrame, int]:
    """The checks `prepare_classification_table` applies; returns the cleaned table and the dropped-row count."""
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("frame must be a pandas.DataFrame")
    if frame.columns.duplicated().any():
        dupes = sorted(set(frame.columns[frame.columns.duplicated()]))
        raise ValueError(f"table contains duplicate column names: {dupes}")
    if target_column not in frame.columns:
        raise KeyError(f"missing target {target_column!r}")
    labelled = frame[target_column].notna()
    dropped = int((~labelled).sum())
    out = frame.loc[labelled].copy().reset_index(drop=True)
    if len(out) < min_rows:
        raise ValueError(f"need at least {min_rows} labelled rows, got {len(out)}")
    features = [column for column in out.columns if column != target_column]
    if not features:
        raise ValueError("No feature columns")
    if len(features) > max_features or len(out) > max_rows:
        raise ValueError(
            f"Operational row/feature ceiling exceeded: rows={len(out)} (MAX_TRAIN_ROWS={max_rows}), "
            f"features={len(features)} (MAX_FEATURES={max_features})"
        )
    if out[target_column].nunique() < min_classes:
        raise ValueError(f"Need at least {min_classes} target classes, got {out[target_column].nunique()}")
    return out, dropped


def prepare_classification_table(
    frame: pd.DataFrame, target_column: str, *, min_rows: int = MIN_TRAIN_ROWS, min_classes: int = MIN_CLASSES
) -> tuple[pd.DataFrame, int]:
    """Drop rows whose target is missing (the count is returned so the notebook can report it — DAT23),
    enforce the row/feature ceilings and the class-count floor."""
    return _check_classification_table(frame, target_column, min_rows=min_rows, min_classes=min_classes)


def check_stratifiable(frame: pd.DataFrame, target_column: str, *, min_rows_per_class: int = MIN_ROWS_PER_CLASS) -> dict[Any, int]:
    """Every class needs at least ``min_rows_per_class`` rows for a stratified split; returns the class counts."""
    counts = frame[target_column].value_counts()
    if counts.size < MIN_CLASSES or int(counts.min()) < min_rows_per_class:
        raise ValueError(
            f"Every class needs >= {min_rows_per_class} rows for a stratified holdout: {counts.to_dict()}"
        )
    return {label: int(n) for label, n in counts.items()}


def align_to_schema(
    frame: pd.DataFrame,
    feature_columns: Sequence[str],
    target_column: str,
    train_classes: Sequence[Any] | None = None,
) -> pd.DataFrame:
    """Require the training schema exactly and (when ``train_classes`` is given) no class unseen in training;
    order columns like train."""
    expected = set(feature_columns) | {target_column}
    if set(frame.columns) != expected:
        raise ValueError(
            f"schema does not match train; missing={sorted(expected - set(frame.columns))}, "
            f"extra={sorted(set(frame.columns) - expected)}"
        )
    if train_classes is not None:
        unseen = sorted(set(frame[target_column]) - set(train_classes), key=str)
        if unseen:
            raise ValueError(f"target classes unseen in training: {unseen}")
    return frame[[*feature_columns, target_column]].reset_index(drop=True)


def fit_categorical_encoder(frame: pd.DataFrame, feature_columns: Sequence[str]) -> dict[str, list[str]]:
    """Ordinal maps for non-numeric columns, fitted on the training split only."""
    encoders: dict[str, list[str]] = {}
    for column in feature_columns:
        series = frame[column]
        if pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):
            continue
        encoders[column] = sorted({str(value) for value in series.dropna().unique()})
    return encoders


def apply_categorical_encoder(
    frame: pd.DataFrame, encoders: Mapping[str, Sequence[str]]
) -> tuple[pd.DataFrame, dict[str, int]]:
    """Apply training-fitted ordinal maps; unseen or missing values get the extra 'unknown' code.

    Returns the encoded frame and, per column, how many unseen values were mapped to the unknown code
    (DAT20: the notebook reports them)."""
    out = frame.copy()
    unseen_counts: dict[str, int] = {}
    for column, categories in encoders.items():
        lookup = {category: index for index, category in enumerate(categories)}
        unknown = len(categories)
        encoded, unseen = [], 0
        for value in out[column]:
            if pd.isna(value):
                encoded.append(unknown)
                continue
            key = str(value)
            if key not in lookup:
                unseen += 1
            encoded.append(lookup.get(key, unknown))
        out[column] = encoded
        if unseen:
            unseen_counts[column] = unseen
    return out, unseen_counts


def _missing_counts(frame: pd.DataFrame) -> dict[str, int]:
    return {str(column): int(n) for column, n in frame.isna().sum().items() if n > 0}


def _check_inference_frame(frame: pd.DataFrame, feature_columns: Sequence[str]) -> pd.DataFrame:
    """The checks `read_inference_csv` applies to an inference table (after the raw-header check)."""
    if frame.columns.duplicated().any():
        dupes = sorted(set(frame.columns[frame.columns.duplicated()]))
        raise ValueError(f"Inference CSV contains duplicate column names: {dupes}")
    if "prediction" in frame.columns or any(str(column).startswith("probability_") for column in frame.columns):
        raise ValueError("Inference CSV already contains prediction/probability output columns")
    missing = [column for column in feature_columns if column not in frame.columns]
    if missing:
        raise ValueError(f"Inference CSV missing features: {missing}")
    return frame


def raw_csv_header(payload: bytes) -> list[str]:
    """First non-empty CSV row, read before pandas can rename duplicate names."""
    reader = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    for row in reader:
        if row and any(cell.strip() for cell in row):
            return row
    raise ValueError("CSV has no header")


def read_csv_payload(payload: bytes, label: str) -> pd.DataFrame:
    """Read a CSV payload, refusing duplicate header names before pandas renames them (DAT16)."""
    header = raw_csv_header(payload)
    dupes = sorted({name for name in header if header.count(name) > 1})
    if dupes:
        raise ValueError(f"{label} contains duplicate column names: {dupes}")
    return pd.read_csv(io.BytesIO(payload))


def read_inference_csv(payload: bytes, feature_columns: Sequence[str]) -> pd.DataFrame:
    """Read an unlabelled inference CSV: unique header, no `prediction`/`probability_*` columns, every feature present."""
    return _check_inference_frame(read_csv_payload(payload, "Inference CSV"), feature_columns)


def classification_metrics(y_true: Any, y_pred: Any, y_proba: Any, classes: Sequence[Any]) -> dict[str, float]:
    """The repository's metric set: accuracy, balanced_accuracy, f1_weighted (discrete correctness under the
    argmax rule), log_loss (probability quality; lower is better) and roc_auc (ranking quality; binary or
    one-vs-rest; NaN when undefined, e.g. a single class present)."""
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        f1_score,
        log_loss,
        roc_auc_score,
    )

    y = np.asarray(y_true).reshape(-1)
    pred = np.asarray(y_pred).reshape(-1)
    proba = np.asarray(y_proba, dtype=float)
    labels = list(classes)
    if y.shape != pred.shape or y.size == 0:
        raise ValueError("y_true and y_pred must be non-empty and the same length")
    if proba.ndim != 2 or proba.shape != (y.size, len(labels)):
        raise ValueError("y_proba must have one row per example and one column per class")
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "f1_weighted": float(f1_score(y, pred, average="weighted")),
        "log_loss": float(log_loss(y, proba, labels=labels)),
    }
    try:
        if len(labels) == 2:
            out["roc_auc"] = float(roc_auc_score(y, proba[:, 1]))
        else:
            out["roc_auc"] = float(roc_auc_score(y, proba, multi_class="ovr", labels=labels))
    except ValueError:
        out["roc_auc"] = float("nan")
    return out


def majority_class_baseline(train_targets: Any, holdout_targets: Any) -> dict[str, float]:
    """The trivial baseline: always predict the most frequent support class, with the support class shares as
    the probabilities (so log_loss is defined; roc_auc is NaN because constant scores rank nothing)."""
    train = pd.Series(np.asarray(train_targets).reshape(-1))
    holdout = np.asarray(holdout_targets).reshape(-1)
    if train.size == 0 or holdout.size == 0:
        raise ValueError("targets must be non-empty")
    shares = train.value_counts(normalize=True).sort_index()
    classes = list(shares.index)
    unseen = sorted(set(holdout) - set(classes), key=str)
    if unseen:
        raise ValueError(f"holdout has target classes unseen in training: {unseen}")
    majority = shares.idxmax()
    proba = np.tile(shares.to_numpy(dtype=float), (holdout.size, 1))
    metrics = classification_metrics(holdout, np.full(holdout.shape, majority, dtype=object), proba, classes)
    metrics["roc_auc"] = float("nan")
    return metrics


def compare_metric(candidate: Mapping[str, float], reference: Mapping[str, float], name: str) -> bool:
    """True when `candidate` beats `reference` on metric `name` (lower log_loss, higher everything else)."""
    candidate_value, reference_value = float(candidate[name]), float(reference[name])
    if not (math.isfinite(candidate_value) and math.isfinite(reference_value)):
        raise ValueError(f"{name} unavailable for selection")
    if name == "log_loss":
        return candidate_value < reference_value
    return candidate_value > reference_value


# ---------------------------------------------------------------------------
# Role stages (DAT24 / EVAL21).
# ---------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "input": "pandas.DataFrame, one row per example; numeric/categorical features plus a categorical target",
    "columns": "unique names; non-numeric feature columns are ordinal-encoded with maps fitted on training",
    "target": (
        "any label type; rows whose target is missing are dropped and counted; at least MIN_CLASSES distinct "
        "classes, and every holdout/test class must be present in the support rows"
    ),
    "train_rows": [MIN_TRAIN_ROWS, MAX_TRAIN_ROWS],
    "eval_rows": [MIN_EVAL_ROWS, None],
    "features": [1, MAX_FEATURES],
    "classes": [MIN_CLASSES, None],
    "rows_per_class_for_stratified_split": [MIN_ROWS_PER_CLASS, None],
    "inference_input": (
        "every fitted feature column present; no `prediction` or `probability_*` column; extras pass through"
    ),
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "no scaling by the package; unseen or missing categorical values map to the fitted 'unknown' code; "
        "the model is conditioned on the (encoded) training rows at prediction time"
    ),
}


def validate_inputs(
    frame: pd.DataFrame,
    target_column: str | None = "target",
    *,
    feature_columns: Sequence[str] | None = None,
    min_rows: int = MIN_TRAIN_ROWS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observed table properties, verdict).

    With a ``target_column`` the table is checked exactly as ``prepare_classification_table`` checks it (the
    number of dropped missing-target rows is recorded, not hidden; class counts are observed); with
    ``target_column=None`` it is an inference table checked against ``feature_columns`` exactly as
    ``read_inference_csv`` checks it. Rejection is reported by raising the same error the core function raises.
    """
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (the table's id)")
    table_id = names[0] if names else "table-0"
    if target_column is None:
        if feature_columns is None:
            raise ValueError("feature_columns is required to validate an inference table")
        checked = _check_inference_frame(frame, list(feature_columns))
        entry: dict[str, Any] = {
            "id": table_id,
            "mode": "inference",
            "rows": len(checked),
            "feature_columns": list(feature_columns),
            "extra_columns": [column for column in checked.columns if column not in feature_columns],
            "missing_value_columns": _missing_counts(checked[list(feature_columns)]),
        }
    else:
        cleaned, dropped = _check_classification_table(frame, target_column, min_rows=min_rows)
        features = [column for column in cleaned.columns if column != target_column]
        encoders = fit_categorical_encoder(cleaned, features)
        counts = cleaned[target_column].value_counts()
        entry = {
            "id": table_id,
            "mode": "fit",
            "rows": len(cleaned),
            "dropped_missing_target_rows": dropped,
            "feature_columns": features,
            "categorical_columns": sorted(encoders),
            "missing_value_columns": _missing_counts(cleaned[features]),
            "classes": [str(label) for label in counts.sort_index().index],
            "class_counts": {str(label): int(n) for label, n in counts.sort_index().items()},
            "majority_class_share": float(counts.max() / counts.sum()),
        }
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [entry],
        "target_column": target_column,
        "min_rows": min_rows if target_column is not None else None,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    metrics: Mapping[str, float] | None,
    *,
    baseline: Mapping[str, float] | None = None,
    independent_test: Mapping[str, float] | None = None,
    n_holdout: int | None = None,
    n_test: int | None = None,
    class_labels: Sequence[Any] | None = None,
    target_column: str | None = None,
    selection: str | None = None,
    sample_kind: str = "sample",
    estimation: str = "single seeded stratified holdout; no dispersion estimate",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``metrics`` / ``independent_test`` are dicts from ``classification_metrics`` and ``baseline`` from
    ``majority_class_baseline``; the verdict is ``sample-sanity``. Without metrics (no labelled rows) the
    verdict is ``not-measurable`` and the report says what labelled data would make the task measurable.
    """

    def _entries(source: Mapping[str, float]) -> list[dict[str, Any]]:
        unknown = sorted(set(source) - set(METRIC_IDS))
        if unknown:
            raise ValueError(f"unknown metric ids {unknown}; classification_metrics reports {list(METRIC_IDS)}")
        return [
            {
                "id": metric_id,
                "value": None if not math.isfinite(float(source[metric_id])) else float(source[metric_id]),
                "units": "nats" if metric_id == "log_loss" else "unitless",
                "higher_is_better": metric_id != "log_loss",
            }
            for metric_id in METRIC_IDS
            if metric_id in source
        ]

    base: dict[str, Any] = {
        "task": "tabular classification by in-context conditioning on labelled support rows",
        "decision_rule": DECISION_RULE,
        "score_semantics": "class probabilities in classes_ order, uncalibrated; no threshold shipped",
        "sample_kind": sample_kind,
        "n_holdout": n_holdout,
        "n_test": n_test,
        "class_labels": None if class_labels is None else [str(label) for label in class_labels],
        "target_column": target_column,
        "selection": selection,
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if metrics is None:
        return {
            **base,
            "metrics": [],
            "independent_test": [],
            "verdict": "not-measurable",
            "reason": "no labelled holdout rows were supplied for the scored table",
            "needs": (
                "a labelled holdout table whose classes all appear in the support rows, scored with "
                "`classification_metrics` (accuracy, balanced_accuracy, f1_weighted, log_loss, roc_auc) against "
                "`majority_class_baseline`; an independent test partition from the deployment domain for any "
                "generalisable claim"
            ),
        }
    reported = [{**entry, "estimation": estimation} for entry in _entries(metrics)]
    test_entries: list[dict[str, Any]] = []
    if independent_test is not None:
        test_estimation = "independent test partition, single run"
        test_entries = [{**e, "estimation": test_estimation} for e in _entries(independent_test)]
    baselines = [] if baseline is None else [{"id": "majority_class", "metrics": _entries(baseline)}]
    rows = "an unstated number of" if n_holdout is None else str(n_holdout)
    return {
        **base,
        "metrics": reported,
        "independent_test": test_entries,
        "baselines": baselines,
        "verdict": "sample-sanity",
        "reason": f"{rows} labelled holdout row(s) from one seeded stratified split; tutorial evidence, not a benchmark",
        "needs": (
            "an independent, domain-representative labelled test set for any generalisable quality claim; the "
            "class probabilities are uncalibrated and any decision threshold must be chosen on the caller's data"
        ),
    }


# ---------------------------------------------------------------------------
# Serving-artifact ZIP handling shared by the producer's fresh-reload check and the companion notebook.
# ---------------------------------------------------------------------------


def safe_extract_zip(
    zip_path: str | Path, dest: str | Path, *, max_expanded_bytes: int = MAX_ARTIFACT_EXPANDED_BYTES
) -> Path:
    """Extract a ZIP member by member after every member passed the path, symlink and size checks (AINF3).

    Members are copied individually (never ``extractall``) so a member that fails a check is never written.
    """
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        expanded_bytes = 0
        for info in archive.infolist():
            if "\\" in info.filename:
                raise ValueError(f"Ambiguous backslash ZIP member: {info.filename}")
            expanded_bytes += info.file_size
            if expanded_bytes > max_expanded_bytes:
                raise ValueError(f"Artifact exceeds {max_expanded_bytes} expanded bytes")
            name = info.filename
            parts = Path(name).parts
            mode = info.external_attr >> 16
            if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode):
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            target = (dest / Path(name)).resolve()
            if root != target and root not in target.parents:
                raise ValueError("ZIP member escapes destination")
        for info in archive.infolist():
            target = (dest / Path(info.filename)).resolve()
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
    return root


def manifest_member_path(root: str | Path, value: Any, field: str) -> Path:
    """Resolve a manifest path inside the bundle root, refusing absolute paths and traversal."""
    rel = Path(str(value))
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe {field} path in artifact.json: {value!r}")
    root_resolved = Path(root).resolve()
    target = (root_resolved / rel).resolve()
    if root_resolved != target and root_resolved not in target.parents:
        raise ValueError(f"{field} path escapes artifact root: {value!r}")
    return target


def verify_artifact_bundle(root: str | Path, manifest: Mapping[str, Any]) -> dict[str, Path]:
    """Check an extracted bundle's allowlist, sizes and digests against its manifest; return member paths."""
    root = Path(root)
    ckpt = manifest_member_path(root, manifest["checkpoint"], "checkpoint")
    context_path = manifest_member_path(root, manifest["trainingContext"], "trainingContext")
    payload_files = manifest.get("payloadFiles")
    if payload_files is not None:
        expected_files = {"artifact.json", *payload_files}
        actual_files = {p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()}
        if actual_files != expected_files:
            unexpected = sorted(actual_files ^ expected_files)
            raise RuntimeError(f"Unexpected or missing artifact files: {unexpected}")
    if manifest.get("sizes"):
        if ckpt.stat().st_size != manifest["sizes"]["checkpoint"]:
            raise RuntimeError("Checkpoint size mismatch")
        if context_path.stat().st_size != manifest["sizes"]["trainingContext"]:
            raise RuntimeError("Training-context size mismatch")
    if sha256_file(ckpt) != manifest["digests"]["checkpointSha256"]:
        raise RuntimeError("Checkpoint digest mismatch")
    if sha256_file(context_path) != manifest["digests"]["trainingContextSha256"]:
        raise RuntimeError("Training-context digest mismatch")
    return {"checkpoint": ckpt, "training_context": context_path}

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `4dcd344ece2c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TabICLClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, n_estimators=8, random_state=42)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "tabicl-classifier-v2",
  "modelId": "jingang/TabICL",
  "revision": "4dcd344ece2c00be9e831fdd35bed57b5ad83e19",
  "files": [
    {
      "path": "tabicl-classifier-v2-20260212.ckpt",
      "bytes": 110368038,
      "sha256": "bdc7dbd5e4ff21f8f0456fcf90c6b7cdf72dbea960f2d05b19bec19f9b3d4ed0"
    }
  ],
  "totalBytes": 110368038
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TabICLClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, n_estimators=8, random_state=42)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Load the sample or your own data

`Sample: Breast Cancer` (default) is the bundled binary sanity check, split 60/20/20 with stratification into support, holdout and an independent test partition; `Sample: Wine` is the bundled three-class numeric table split the same way; `Sample: Palmer Penguins` is the repository's pinned CC0 archive with mixed categorical/numeric features and its own train/val/test split, fetched by immutable commit and digest-verified before it is read (see `examples/sample-data/DATASET_CARD.md`). `Upload CSV` takes one CSV with a `target` column and carves a stratified 80/20 holdout (`check_stratifiable` requires at least `MIN_ROWS_PER_CLASS` rows per class); `Upload pre-split train/val/test` takes your own partitions (`test.csv` optional). Rows whose target is missing are **dropped and counted** (reported in the input manifest of Section 5, never hidden). Non-numeric feature columns are ordinal-encoded with maps fitted on the support split only; unseen or missing values map to an extra 'unknown' code and are counted.

**BYOD privacy boundary.** Uploaded CSV bytes are read inside the current notebook runtime and are not sent by this notebook to an external inference or training service; the network requests on the default path are the pinned checkpoint download of Section 3 and, for `Sample: Palmer Penguins`, one pinned archive.

In [ ]:
import hashlib
import urllib.request

from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split

DATA_SOURCE = 'Sample: Breast Cancer'  # @param ["Sample: Breast Cancer", "Sample: Wine", "Sample: Palmer Penguins", "Upload CSV", "Upload pre-split train/val/test"]
USE_BYOD = False  # @param {type:"boolean"}
TARGET_COLUMN = 'target'
VALIDATION_SPLIT = 0.20
RANDOM_SEED = 42
NETWORK_TIMEOUT_SECONDS = 30
if USE_BYOD and DATA_SOURCE.startswith('Sample'):
    raise ValueError('USE_BYOD=True requires DATA_SOURCE = "Upload CSV" or "Upload pre-split train/val/test".')
if DATA_SOURCE.startswith('Upload') and not USE_BYOD:
    raise ValueError('Set USE_BYOD=True to use an upload DATA_SOURCE.')

def stratified_60_20_20(frame):
    train, remainder = train_test_split(frame, test_size=0.4, random_state=RANDOM_SEED, stratify=frame[TARGET_COLUMN])
    holdout, test = train_test_split(remainder, test_size=0.5, random_state=RANDOM_SEED, stratify=remainder[TARGET_COLUMN])
    return train, holdout, test

test_data = None
if DATA_SOURCE == 'Sample: Breast Cancer':
    dataset = load_breast_cancer(as_frame=True)
    frame = dataset.frame.rename(columns={dataset.target.name: TARGET_COLUMN})
    frame[TARGET_COLUMN] = frame[TARGET_COLUMN].map({0: 'malignant', 1: 'benign'})
    train_data, holdout_data, test_data = stratified_60_20_20(frame)
    data_name, sample_kind = 'sklearn-breast-cancer', 'sample'
elif DATA_SOURCE == 'Sample: Wine':
    dataset = load_wine(as_frame=True)
    frame = dataset.frame.rename(columns={dataset.target.name: TARGET_COLUMN})
    frame[TARGET_COLUMN] = frame[TARGET_COLUMN].map(dict(enumerate(dataset.target_names)))
    train_data, holdout_data, test_data = stratified_60_20_20(frame)
    data_name, sample_kind = 'sklearn-wine', 'sample'
elif DATA_SOURCE == 'Sample: Palmer Penguins':
    TARGET_COLUMN = 'species'
    # Deterministic archive built by examples/build_sample_datasets.py at an immutable commit; digest recorded in DATASET_CARD.md.
    # Optional, non-default source: the archive is fetched from the repository's raw content at an immutable commit (not a clone, not a package install) and digest-verified; the default sample path needs no repository access.
    sample_url = 'https://raw.githubusercontent.com/kurtvalcorza/tabicl-classifier-pipeline/169e60fa8d956aa389c144ac9c7988b92db84a79/examples/sample-data/palmer-penguins.zip'
    SAMPLE_ARCHIVE_SHA256 = 'fe894295ccc0a447dc020f5d57798968b95eae3da1a4cbe09a638e6b1bd0ba44'
    with urllib.request.urlopen(sample_url, timeout=NETWORK_TIMEOUT_SECONDS) as response:
        sample_payload = response.read()
    sample_digest = hashlib.sha256(sample_payload).hexdigest()
    if sample_digest != SAMPLE_ARCHIVE_SHA256:
        raise ValueError(f'Palmer Penguins sample archive digest mismatch: {sample_digest} != {SAMPLE_ARCHIVE_SHA256}')
    sample_zip = Path('work') / 'palmer-penguins.zip'
    sample_zip.parent.mkdir(parents=True, exist_ok=True)
    sample_zip.write_bytes(sample_payload)
    sample_root = safe_extract_zip(sample_zip, Path('work') / 'palmer-penguins')  # member-by-member, after path/symlink/size checks
    names = {path.name: path for path in sample_root.rglob('*.csv')}
    if not {'train.csv', 'val.csv', 'test.csv'} <= set(names):
        raise ValueError('Palmer Penguins sample archive is missing train.csv, val.csv, or test.csv')
    train_data = read_csv_payload(names['train.csv'].read_bytes(), 'train.csv')
    holdout_data = read_csv_payload(names['val.csv'].read_bytes(), 'val.csv')
    test_data = read_csv_payload(names['test.csv'].read_bytes(), 'test.csv')
    data_name, sample_kind = 'palmer-penguins-cc0', 'sample'
elif DATA_SOURCE == 'Upload CSV':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one CSV')
    data_name, payload = next(iter(uploaded.items()))
    frame, _dropped = prepare_classification_table(read_csv_payload(payload, data_name), TARGET_COLUMN)
    check_stratifiable(frame, TARGET_COLUMN)
    train_data, holdout_data = train_test_split(frame, test_size=VALIDATION_SPLIT, random_state=RANDOM_SEED, stratify=frame[TARGET_COLUMN])
    sample_kind = 'BYOD'
else:
    from google.colab import files
    uploaded = files.upload()
    by_name = {Path(key).name.lower(): (key, value) for key, value in uploaded.items()}
    if not {'train.csv', 'val.csv'} <= set(by_name):
        raise ValueError('Upload train.csv and val.csv; test.csv optional')
    train_data = read_csv_payload(by_name['train.csv'][1], 'train.csv')
    holdout_data = read_csv_payload(by_name['val.csv'][1], 'val.csv')
    if 'test.csv' in by_name:
        test_data = read_csv_payload(by_name['test.csv'][1], 'test.csv')
    data_name, sample_kind = 'pre-split upload', 'BYOD'

sample_sha256 = hashlib.sha256(pd.concat([train_data, holdout_data] + ([test_data] if test_data is not None else [])).to_csv(index=False).encode('utf-8')).hexdigest()
print({'sample_kind': sample_kind, 'name': data_name, 'target': TARGET_COLUMN, 'train_rows': len(train_data), 'holdout_rows': len(holdout_data), 'test_rows': 0 if test_data is None else len(test_data), 'csv_sha256': sample_sha256})

## 5. Validate the tables → input manifest, then encode

`validate_inputs` is the package's public validation stage: it applies exactly the checks `prepare_classification_table` applies — unique column names, the target present, missing-target rows dropped with the count recorded, at least `MIN_TRAIN_ROWS` support rows (`MIN_EVAL_ROWS` for a holdout), at most `MAX_TRAIN_ROWS` rows and `MAX_FEATURES` columns, at least `MIN_CLASSES` classes — and returns an **input manifest** naming the schema, the observed structure (categorical columns, missing values, the class list and class counts, the majority-class share) and the verdict. It is written to `outputs/tabiclv2_classifier_input_manifest.json`. To show what rejection looks like, the cell also validates a probe with too few rows and records the package's own error message as a finding. The ceilings and the decision rule are printed before any model runs.

The holdout and independent test partitions are then aligned to the support schema **and to the support class set** (`align_to_schema` refuses a class the model never saw — an in-context classifier can only predict classes present in its context), the categorical encoder is fitted on the support split only and applied everywhere (unseen values counted), and the trivial baseline is computed with `majority_class_baseline`: always predict the most frequent support class. Its accuracy is the majority-class share printed here, and everything in Section 6 should be read against it.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_TRAIN_ROWS': MIN_TRAIN_ROWS, 'MIN_EVAL_ROWS': MIN_EVAL_ROWS, 'MAX_TRAIN_ROWS': MAX_TRAIN_ROWS, 'MAX_FEATURES': MAX_FEATURES, 'MIN_CLASSES': MIN_CLASSES, 'MIN_ROWS_PER_CLASS': MIN_ROWS_PER_CLASS}, 'decision_rule': DECISION_RULE})
input_manifest = validate_inputs(train_data, target_column=TARGET_COLUMN, names=[data_name + ':train'])
holdout_manifest = validate_inputs(holdout_data, target_column=TARGET_COLUMN, min_rows=MIN_EVAL_ROWS, names=[data_name + ':holdout'])
input_manifest['inputs'].extend(holdout_manifest['inputs'])
if test_data is not None:
    input_manifest['inputs'].extend(validate_inputs(test_data, target_column=TARGET_COLUMN, min_rows=MIN_EVAL_ROWS, names=[data_name + ':test'])['inputs'])
# Demonstrate rejection on a probe that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(train_data.head(MIN_TRAIN_ROWS - 1), target_column=TARGET_COLUMN)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'too-few-rows-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/tabiclv2_classifier_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest['inputs'][0], indent=2))
print('findings:', input_manifest['findings'])

train_data, dropped_train = prepare_classification_table(train_data, TARGET_COLUMN)
holdout_data, dropped_holdout = prepare_classification_table(holdout_data, TARGET_COLUMN, min_rows=MIN_EVAL_ROWS, min_classes=1)
dropped_test = 0
if test_data is not None:
    test_data, dropped_test = prepare_classification_table(test_data, TARGET_COLUMN, min_rows=MIN_EVAL_ROWS, min_classes=1)
FEATURE_COLUMNS = [column for column in train_data.columns if column != TARGET_COLUMN]
TRAIN_CLASSES = sorted(train_data[TARGET_COLUMN].unique(), key=str)
train_data = train_data[FEATURE_COLUMNS + [TARGET_COLUMN]].reset_index(drop=True)
holdout_data = align_to_schema(holdout_data, FEATURE_COLUMNS, TARGET_COLUMN, TRAIN_CLASSES)
test_data = align_to_schema(test_data, FEATURE_COLUMNS, TARGET_COLUMN, TRAIN_CLASSES) if test_data is not None else None
CATEGORICAL_ENCODERS = fit_categorical_encoder(train_data, FEATURE_COLUMNS)
train_encoded, _ = apply_categorical_encoder(train_data, CATEGORICAL_ENCODERS)
holdout_encoded, unseen_holdout = apply_categorical_encoder(holdout_data, CATEGORICAL_ENCODERS)
test_encoded, unseen_test = (apply_categorical_encoder(test_data, CATEGORICAL_ENCODERS) if test_data is not None else (None, {}))
baseline = majority_class_baseline(train_encoded[TARGET_COLUMN], holdout_encoded[TARGET_COLUMN])
class_shares = train_data[TARGET_COLUMN].value_counts(normalize=True).round(3).to_dict()
print({'dropped_missing_target_rows': {'train': dropped_train, 'holdout': dropped_holdout, 'test': dropped_test}, 'unseen_categorical_values': {'holdout': unseen_holdout, 'test': unseen_test}, 'encoded_categoricals': sorted(CATEGORICAL_ENCODERS)})
print({'train': len(train_encoded), 'holdout': len(holdout_encoded), 'test': 0 if test_encoded is None else len(test_encoded), 'features': len(FEATURE_COLUMNS), 'classes': TRAIN_CLASSES, 'training_class_shares': class_shares})
print('majority-class baseline on the holdout', baseline)

## 6. Evaluate pretrained TabICLv2, then optionally fine-tune

`pipe.fit(X_train, y_train)` does not train anything; it registers the support rows as the model's context (the checkpoint pinned in Section 3 is deserialised by `tabicl` here). `n_estimators=8` averages eight passes with different feature/row permutations, which is why the same checkpoint gives smoother probabilities than a single pass. `classification_metrics` scores the holdout and, when present, the independent test partition: **accuracy**, **balanced accuracy** and **weighted F1** are discrete correctness under the implicit `argmax` rule; **log loss** scores the class probabilities and penalises confident mistakes (a perfect model scores 0); **ROC-AUC** (binary, or one-vs-rest for more classes) scores ranking quality independent of any threshold and is `NaN` when undefined. Reading accuracy alone hides both imbalance and probability-quality failures.

**Fine-tuning gate (off by default; CUDA only).** With `RUN_FINE_TUNING=True`, `create_finetuned_classifier` runs the upstream gradient fine-tuning with early stopping on the holdout and `EVAL_METRIC`, the best checkpoint is reloaded into an ordinary classifier with the same inference ensemble for a fair comparison, and the candidate replaces the pretrained model **only** if it beats it on the holdout (`compare_metric`) and the holdout has at least `MIN_SELECTION_HOLDOUT_ROWS` rows. The independent test is evidence only; a worse test result is surfaced as a warning and never changes the selection. On the Breast Cancer sample the fine-tuned lines typically equal the pretrained ones: the pretrained model is already near the ceiling there.

**Fine-tuning disk usage.** TabICL writes an epoch checkpoint per epoch, so temporary disk use scales with `FINE_TUNE_EPOCHS`; after the best one is loaded and evaluated, the cell deletes the others and keeps `best.ckpt` only. The fine-tuned `best.ckpt` may remain larger than the base checkpoint because upstream training state can be embedded; the notebook preserves the upstream checkpoint format for checkpoint compatibility rather than rewriting serialised state.

**Reproducibility boundary.** The tutorial fixes the split seed and the TabICL ensemble seed. Those controls make the demonstrated partitioning and estimator configuration repeatable, but they do not promise bitwise-identical floating-point results across devices, library builds or kernel choices.

In [ ]:
import shutil

RUN_FINE_TUNING = False  # @param {type:"boolean"}
EVAL_METRIC = 'accuracy'  # @param ["accuracy", "log_loss", "roc_auc"]
FINE_TUNE_EPOCHS, FINE_TUNE_TIME_LIMIT, FINE_TUNE_PATIENCE = 10, 600, 3
MIN_SELECTION_HOLDOUT_ROWS = 50
if EVAL_METRIC not in {'accuracy', 'log_loss', 'roc_auc'}:
    raise ValueError(f'Unsupported EVAL_METRIC: {EVAL_METRIC}')
X_train, y_train = train_encoded[FEATURE_COLUMNS], train_encoded[TARGET_COLUMN]

def score(model, frame):
    X = frame[FEATURE_COLUMNS]
    return classification_metrics(frame[TARGET_COLUMN].to_numpy(), model.predict(X), model.predict_proba(X), model.classes_)

pipe.fit(X_train, y_train)
if list(pipe.classes_) != TRAIN_CLASSES:
    print({'note': 'estimator class order differs from the sorted support classes; probabilities are reported in estimator order', 'classes_': pipe.classes_})
pretrained_metrics = score(pipe, holdout_encoded)
pretrained_test_metrics = score(pipe, test_encoded) if test_encoded is not None else None
if not math.isfinite(pretrained_metrics[EVAL_METRIC]):
    raise ValueError(f'{EVAL_METRIC} is unavailable on the holdout before fine-tuning; choose another selection metric or provide a holdout with sufficient class coverage')
print('pretrained holdout', pretrained_metrics)
if pretrained_test_metrics:
    print('pretrained independent test', pretrained_test_metrics)

candidate = candidate_metrics = candidate_test_metrics = candidate_checkpoint = None
if RUN_FINE_TUNING:
    if not torch.cuda.is_available():
        raise RuntimeError('TabICLv2 fine-tuning requires CUDA')
    ft_dir = Path('outputs') / 'finetune'
    if ft_dir.exists():
        shutil.rmtree(ft_dir)
    finetuner = create_finetuned_classifier(epochs=FINE_TUNE_EPOCHS, learning_rate=1e-5, weight_decay=0.01, n_estimators_finetune=1, n_estimators_validation=1, n_estimators_inference=4, early_stopping=True, patience=FINE_TUNE_PATIENCE, time_limit=FINE_TUNE_TIME_LIMIT, eval_metric=EVAL_METRIC, model_path=str(pipe.model_path), allow_auto_download=False, device='cuda', random_state=RANDOM_SEED, verbose=True, support_many_classes=True)
    fine_tune_classifier(finetuner, X_train, y_train, X_val=holdout_encoded[FEATURE_COLUMNS], y_val=holdout_encoded[TARGET_COLUMN], output_dir=str(ft_dir))
    candidate_checkpoint = ft_dir / 'best.ckpt'
    if not candidate_checkpoint.exists():
        raise RuntimeError('Fine-tuning did not produce best.ckpt')
    # Reload into the ordinary classifier with the pretrained model's inference ensemble for a fair comparison.
    candidate = TabICLClassificationPipeline(create_classifier(model_path=candidate_checkpoint, allow_auto_download=False, n_estimators=pipe.n_estimators, random_state=RANDOM_SEED, device=pipe.device), model_path=candidate_checkpoint, n_estimators=pipe.n_estimators, random_state=RANDOM_SEED, device=pipe.device, source='fine-tuned')
    candidate.fit(X_train, y_train)
    candidate_metrics = score(candidate, holdout_encoded)
    candidate_test_metrics = score(candidate, test_encoded) if test_encoded is not None else None
    print('candidate holdout', candidate_metrics)
    for checkpoint in [c for c in ft_dir.rglob('*.ckpt') if c.resolve() != candidate_checkpoint.resolve()]:
        checkpoint.unlink()

ACTIVE_MODEL, ACTIVE_CHECKPOINT_PATH, ACTIVE_MODE = pipe, pipe.model_path, 'pretrained'
SELECTION_BASIS = 'default:pretrained'
if candidate is not None:
    if len(holdout_encoded) < MIN_SELECTION_HOLDOUT_ROWS:
        SELECTION_BASIS = f'default:pretrained; holdout-too-small:{len(holdout_encoded)}<{MIN_SELECTION_HOLDOUT_ROWS}'
    else:
        SELECTION_BASIS = f'holdout:{EVAL_METRIC}'
        if compare_metric(candidate_metrics, pretrained_metrics, EVAL_METRIC):
            ACTIVE_MODEL, ACTIVE_CHECKPOINT_PATH, ACTIVE_MODE = candidate, candidate_checkpoint, 'fine-tuned'
    if candidate_test_metrics and pretrained_test_metrics:
        degraded = [name for name in METRIC_IDS if math.isfinite(candidate_test_metrics[name]) and math.isfinite(pretrained_test_metrics[name]) and not compare_metric(candidate_test_metrics, pretrained_test_metrics, name) and candidate_test_metrics[name] != pretrained_test_metrics[name]]
        if degraded:
            print('WARNING independent-test metrics worsened for the candidate:', degraded, '(evidence only; never used for selection)')
active_metrics = candidate_metrics if ACTIVE_MODE == 'fine-tuned' else pretrained_metrics
active_test_metrics = candidate_test_metrics if ACTIVE_MODE == 'fine-tuned' else pretrained_test_metrics
CLASSES = list(ACTIVE_MODEL.classes_)
print({'recommended_for_export': ACTIVE_MODE, 'selection_basis': SELECTION_BASIS, 'source': ACTIVE_MODEL.source, 'device': ACTIVE_MODEL.device, 'classes_': CLASSES})

## 7. Classical tree baselines and in-memory blending, then the evaluation report

LightGBM and Random Forest are fitted on the exact same encoded support rows and scored on the exact same holdout and test partitions, so the comparison is fair (EVAL15). Their probability columns are mapped onto the TabICLv2 class order with `align_probabilities` **before** any blending, so a multiclass label permutation can never pass silently. A convex blend of TabICLv2 and LightGBM probabilities is then chosen by maximising holdout accuracy over a 101-point grid (ties broken toward the foundation model by taking the upper-median tied weight); it is evaluated in memory only, reported next to the one-row resolution ($1/N$) so small differences are read honestly, and the exported bundle in Section 9 is the unchanged active model. Latencies are medians of five warmed runs on this runtime.

`evaluation_report` is the package's public evaluation stage and always produces a report. Here it carries the active model's holdout metrics (`accuracy`, `balanced_accuracy`, `f1_weighted`, `log_loss`, `roc_auc` — the repository's own metric ids), the independent-test metrics when a test partition exists, and the majority-class baseline, with the verdict `sample-sanity`: one seeded stratified split with no dispersion estimate, tutorial evidence rather than a benchmark. Without a labelled holdout the verdict would be `not-measurable`. The report is written to `outputs/tabiclv2_classifier_evaluation_report.json`.

In [ ]:
import time

from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

lgbm_model = LGBMClassifier(random_state=RANDOM_SEED, n_estimators=100, verbose=-1).fit(X_train, y_train)
rf_model = RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=100).fit(X_train, y_train)

def timed_predict_proba(model, frame, repeats=5):
    X = frame[FEATURE_COLUMNS]
    if repeats > 1:
        model.predict_proba(X)  # discarded warm-up call
    latencies = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        raw = np.asarray(model.predict_proba(X), dtype=float)
        latencies.append((time.perf_counter() - t0) * 1000.0)
    proba = align_probabilities(raw, list(model.classes_), CLASSES)
    return proba, np.asarray(CLASSES, dtype=object)[np.argmax(proba, axis=1)], float(np.median(latencies))

y_holdout = holdout_encoded[TARGET_COLUMN].to_numpy()
n_holdout = len(holdout_encoded)
prob_tabicl_holdout, pred_tabicl_holdout, lat_tabicl = timed_predict_proba(ACTIVE_MODEL, holdout_encoded)
prob_lgbm_holdout, pred_lgbm_holdout, lat_lgbm = timed_predict_proba(lgbm_model, holdout_encoded)
prob_rf_holdout, pred_rf_holdout, lat_rf = timed_predict_proba(rf_model, holdout_encoded)
classical = {'tabicl_' + ACTIVE_MODE: classification_metrics(y_holdout, pred_tabicl_holdout, prob_tabicl_holdout, CLASSES), 'lightgbm': classification_metrics(y_holdout, pred_lgbm_holdout, prob_lgbm_holdout, CLASSES), 'random_forest': classification_metrics(y_holdout, pred_rf_holdout, prob_rf_holdout, CLASSES)}
print({'holdout_rows': n_holdout, 'one_row_resolution_pct': round(100.0 / n_holdout, 2)})
for name, values in classical.items():
    print(f"{name:<20} accuracy={values['accuracy']:.4f} roc_auc={values['roc_auc']:.4f} log_loss={values['log_loss']:.4f}")
print({'latency_ms_median_of_5': {'tabicl': round(lat_tabicl, 2), 'lightgbm': round(lat_lgbm, 2), 'random_forest': round(lat_rf, 2)}, 'rows': n_holdout, 'device': ACTIVE_MODEL.device})

best_acc, tied_weights = -1.0, []
for w in np.linspace(0.0, 1.0, 101):
    blend_p = w * prob_tabicl_holdout + (1.0 - w) * prob_lgbm_holdout
    acc = float(np.mean(np.asarray(CLASSES, dtype=object)[np.argmax(blend_p, axis=1)] == y_holdout))
    if acc > best_acc + 1e-9:
        best_acc, tied_weights = acc, [float(w)]
    elif abs(acc - best_acc) <= 1e-9:
        tied_weights.append(float(w))
best_w = tied_weights[len(tied_weights) // 2]  # upper-median tied weight: even ties break toward the foundation model
prob_blend_holdout = best_w * prob_tabicl_holdout + (1.0 - best_w) * prob_lgbm_holdout
pred_blend_holdout = np.asarray(CLASSES, dtype=object)[np.argmax(prob_blend_holdout, axis=1)]
blend_holdout = classification_metrics(y_holdout, pred_blend_holdout, prob_blend_holdout, CLASSES)
print({'blend_objective': 'maximise holdout accuracy', 'weight_tabicl': best_w, 'tied_weights': len(tied_weights), 'blend_holdout': blend_holdout, 'accuracy_delta_vs_tabicl': round(blend_holdout['accuracy'] - classical['tabicl_' + ACTIVE_MODE]['accuracy'], 4)})
if len(tied_weights) > 1:
    print(f'Holdout accuracy is saturated across {len(tied_weights)} grid weights; the selected blend weight is not uniquely informative.')
blend_test = None
if test_encoded is not None:
    y_test = test_encoded[TARGET_COLUMN].to_numpy()
    prob_tabicl_test, pred_tabicl_test, _ = timed_predict_proba(ACTIVE_MODEL, test_encoded, repeats=1)
    prob_lgbm_test, pred_lgbm_test, _ = timed_predict_proba(lgbm_model, test_encoded, repeats=1)
    prob_blend_test = best_w * prob_tabicl_test + (1.0 - best_w) * prob_lgbm_test
    tabicl_test = classification_metrics(y_test, pred_tabicl_test, prob_tabicl_test, CLASSES)
    blend_test = classification_metrics(y_test, np.asarray(CLASSES, dtype=object)[np.argmax(prob_blend_test, axis=1)], prob_blend_test, CLASSES)
    print({'independent_test': {'tabicl': tabicl_test, 'lightgbm': classification_metrics(y_test, pred_lgbm_test, prob_lgbm_test, CLASSES), 'blend': blend_test}})
    if blend_holdout['accuracy'] > classical['tabicl_' + ACTIVE_MODE]['accuracy'] and blend_test['accuracy'] <= tabicl_test['accuracy']:
        print('WARNING mixed evidence: the blend improved holdout accuracy but not independent-test accuracy; treat the holdout gain as selection-biased.')
else:
    print('No independent test partition; the holdout blend score is selection-biased demonstration evidence.')

report = evaluation_report(active_metrics, baseline=baseline, independent_test=active_test_metrics, n_holdout=n_holdout, n_test=None if test_encoded is None else len(test_encoded), class_labels=CLASSES, target_column=TARGET_COLUMN, selection=SELECTION_BASIS, sample_kind=sample_kind, estimation='single seeded stratified split (support/holdout/independent test); no dispersion estimate')
report['classical_baselines_holdout'] = {name: values for name, values in classical.items() if not name.startswith('tabicl_')}
report['blend_holdout'] = {'weight_tabicl': best_w, **blend_holdout}
with open('outputs/tabiclv2_classifier_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({key: report[key] for key in ('verdict', 'reason', 'decision_rule', 'selection', 'n_holdout', 'n_test')}, indent=2))

## 8. Optional new-data inference with class probabilities

Off by default so a top-to-bottom run needs no upload dialog. Switch `RUN_NEW_DATA_INFERENCE` on and upload one CSV with the support feature columns (order does not matter; extra columns are preserved in the output and not passed to the model), or set `NEW_DATA_PATH` for a non-interactive executor. `validate_inputs(..., target_column=None, feature_columns=...)` applies exactly the checks `read_inference_csv` applies — unique header, no pre-existing `prediction`/`probability_*` columns, every feature present — and the same fitted categorical encoder is applied (unseen values counted). The output adds `prediction` (the `argmax` label) and one `probability_<class>` column per class **in the fitted class order**; the probabilities are uncalibrated and the package ships no threshold, so a cost-sensitive decision must threshold the relevant probability on the caller's own labelled data.

In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}
NEW_DATA_PATH = ''  # @param {type:"string"}

def with_predictions(raw_rows, encoded_rows):
    out = raw_rows.copy()
    proba = ACTIVE_MODEL.predict_proba(encoded_rows)
    out['prediction'] = ACTIVE_MODEL.predict(encoded_rows)
    for index, class_label in enumerate(CLASSES):
        out[f'probability_{class_label}'] = proba[:, index]
    return out

new_data_result = None
if RUN_NEW_DATA_INFERENCE:
    if NEW_DATA_PATH:
        input_name, payload = os.path.basename(NEW_DATA_PATH), Path(NEW_DATA_PATH).read_bytes()
    else:
        from google.colab import files
        new_upload = files.upload()
        if len(new_upload) != 1:
            raise ValueError('Upload exactly one CSV')
        input_name, payload = next(iter(new_upload.items()))
    rows = read_inference_csv(payload, FEATURE_COLUMNS)
    inference_manifest = validate_inputs(rows, None, feature_columns=FEATURE_COLUMNS, names=[input_name])
    X_new, unseen_new = apply_categorical_encoder(rows[FEATURE_COLUMNS], CATEGORICAL_ENCODERS)
    out = with_predictions(rows, X_new)
    out.to_csv('outputs/tabiclv2_classifier_predictions.csv', index=False)
    new_data_result = {'input': input_name, 'rows': len(out), 'unseen_categorical_values': unseen_new, 'input_manifest': inference_manifest}
    print(out.head())
else:
    # Sample path: score eight held-out rows so a prediction CSV always exists.
    out = with_predictions(holdout_data[FEATURE_COLUMNS].head(8), holdout_encoded[FEATURE_COLUMNS].head(8))
    out.to_csv('outputs/tabiclv2_classifier_predictions.csv', index=False)
    print('Inference upload skipped; eight held-out rows scored instead.')
    print(out)

## 9. Export a DIMER-style serving bundle, then prove a fresh reload

The ZIP carries the minimum serving contract the DIMER pipeline expects: `checkpoints/best.ckpt`, `training_context.parquet`, and `artifact.json`. The training context is *required*: TabICL is still an in-context learner at serve time, so whoever loads the bundle must hand the model the same rows you evaluated with (ART3), and the context inherits the source data's confidentiality, licensing, retention and disclosure obligations (ART7). `artifact.json` records the feature and target columns, the class labels, the base-model identity and digest, the selected mode and its basis, the inference settings including the fitted categorical encoders, per-file sizes and SHA-256 digests, the payload allowlist and the producer runtime.

Before calling the ZIP reusable, the cell extracts it into a fresh directory with `safe_extract_zip` (member-by-member, after path, symlink and expanded-size checks — the same function the companion notebook applies), verifies the allowlist, sizes and digests with `verify_artifact_bundle`, rebuilds the classifier from the bundle alone (checkpoint + context + recorded settings), and checks that its labels equal and its probabilities agree with the in-memory model's on eight held-out rows (VER1–VER5). The result JSON then records everything: predictions, metrics, the evaluation report, the input manifest, the sample digest, the notebook's source, the model identity, revision and licence, and the runtime identity.

In [ ]:
ARTIFACT_DIR = Path('outputs') / 'artifact'
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
(ARTIFACT_DIR / 'checkpoints').mkdir(parents=True)
export_ckpt = ARTIFACT_DIR / 'checkpoints' / 'best.ckpt'
shutil.copy2(ACTIVE_CHECKPOINT_PATH, export_ckpt)
context_path = ARTIFACT_DIR / 'training_context.parquet'
train_encoded[FEATURE_COLUMNS + [TARGET_COLUMN]].to_parquet(context_path, index=False)
manifest = {
    'artifactFormat': ARTIFACT_FORMAT,
    'checkpoint': 'checkpoints/best.ckpt', 'trainingContext': 'training_context.parquet',
    'targetColumn': TARGET_COLUMN, 'featureColumns': FEATURE_COLUMNS, 'classLabels': [str(label) for label in CLASSES],
    'baseCheckpoint': BASE_CHECKPOINT_NAME, 'baseModelRevision': MODEL_REVISION, 'baseModelSha256': BASE_MODEL_SHA256,
    'tabiclVersion': importlib.metadata.version('tabicl'), 'mode': ACTIVE_MODE, 'selectionBasis': SELECTION_BASIS,
    'metrics': {'selectionMetric': EVAL_METRIC, 'pretrainedHoldout': pretrained_metrics, 'fineTunedHoldout': candidate_metrics, 'pretrainedIndependentTest': pretrained_test_metrics, 'fineTunedIndependentTest': candidate_test_metrics},
    'inference': {'class': 'TabICLClassifier', 'modelPath': 'checkpoints/best.ckpt', 'nEstimators': ACTIVE_MODEL.n_estimators, 'randomState': RANDOM_SEED, 'supportManyClasses': True, 'allowAutoDownload': False, 'decisionRule': DECISION_RULE, 'categoricalEncoders': CATEGORICAL_ENCODERS},
    'digests': {'checkpointSha256': sha256_file(export_ckpt), 'trainingContextSha256': sha256_file(context_path)},
    'sizes': {'checkpoint': export_ckpt.stat().st_size, 'trainingContext': context_path.stat().st_size},
    'payloadFiles': ['checkpoints/best.ckpt', 'training_context.parquet'],
    'runtime': {'pythonVersion': platform.python_version(), 'torchVersion': torch.__version__, 'pandasVersion': pandas.__version__, 'pyarrowVersion': importlib.metadata.version('pyarrow'), 'scikitLearnVersion': sklearn.__version__, 'device': ACTIVE_MODEL.device},
    'notebookSource': NOTEBOOK_SOURCE,
}
(ARTIFACT_DIR / 'artifact.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
archive_path = Path(shutil.make_archive(str(Path('outputs') / 'tabiclv2_classifier_artifact'), 'zip', root_dir=ARTIFACT_DIR))
print({'artifact_zip': str(archive_path), 'zip_sha256': sha256_file(archive_path)})

RELOAD_DIR = Path('outputs') / 'artifact-reload'
if RELOAD_DIR.exists():
    shutil.rmtree(RELOAD_DIR)
reload_root = safe_extract_zip(archive_path, RELOAD_DIR)
served = json.loads((reload_root / 'artifact.json').read_text(encoding='utf-8'))
members = verify_artifact_bundle(reload_root, served)
context = pd.read_parquet(members['training_context'])
reloaded = TabICLClassificationPipeline(create_classifier(model_path=members['checkpoint'], allow_auto_download=False, n_estimators=served['inference']['nEstimators'], random_state=served['inference']['randomState'], device=ACTIVE_MODEL.device), model_path=members['checkpoint'], n_estimators=served['inference']['nEstimators'], random_state=served['inference']['randomState'], device=ACTIVE_MODEL.device, source='artifact')
reloaded.fit(context[served['featureColumns']], context[served['targetColumn']])
smoke_rows = holdout_encoded[FEATURE_COLUMNS].iloc[:min(8, len(holdout_encoded))]
if list(reloaded.classes_) != CLASSES:
    raise RuntimeError('Reloaded class labels differ from the exported class order')
if not np.array_equal(ACTIVE_MODEL.predict(smoke_rows), reloaded.predict(smoke_rows)):
    raise RuntimeError('Prediction mismatch')
np.testing.assert_allclose(ACTIVE_MODEL.predict_proba(smoke_rows), reloaded.predict_proba(smoke_rows), rtol=1e-5, atol=1e-7)
print('PASS: bundle extracted safely, digests verified, classifier rebuilt from the bundle alone; labels identical and probabilities equivalent (rtol=1e-5, atol=1e-7).')

payload = {
    'predictions': out.to_dict(orient='records'),
    'new_data': new_data_result,
    'metrics': {'active_mode': ACTIVE_MODE, 'holdout': active_metrics, 'independent_test': active_test_metrics, 'pretrained_holdout': pretrained_metrics, 'candidate_holdout': candidate_metrics},
    'majority_class_baseline': baseline,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'inference': {'decisionRule': DECISION_RULE + ' over uncalibrated class probabilities', 'threshold': None, 'classLabels': [str(label) for label in CLASSES]},
    'sample': {'kind': sample_kind, 'name': data_name, 'source': DATA_SOURCE, 'csv_sha256': sample_sha256, 'train_rows': len(train_encoded), 'holdout_rows': n_holdout, 'test_rows': 0 if test_encoded is None else len(test_encoded), 'training_class_shares': class_shares},
    'artifact': {'zip': archive_path.name, 'zip_sha256': sha256_file(archive_path), 'manifest': manifest},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'tabicl': importlib.metadata.version('tabicl'), 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'device': ACTIVE_MODEL.device},
}
with open('outputs/tabiclv2_classifier_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

`prediction` is the `argmax` of uncalibrated class probabilities over the fitted class set; the package ships no threshold, and any deployment cut-off must be chosen on the caller's own labelled, domain-representative data. The evaluation report's `sample-sanity` verdict names what it is: one seeded stratified split of a public sample with no dispersion estimate — tutorial evidence that must not be generalised. On the Breast Cancer sample a development run of the previous notebook revision gave pretrained holdout accuracy ≈ 0.974 (test ≈ 0.956), log loss ≈ 0.038 (test ≈ 0.085) and ROC-AUC ≈ 0.9997 (test ≈ 0.995) against a majority-class baseline of about 0.63; the test numbers are a little worse than the holdout's, which is the normal cost of measuring on rows the selection never looked at and why the independent test exists and never drives selection. The blend weight is chosen on the holdout and is therefore selection-biased; the classical baselines show when the foundation model adds value and what it costs in latency. Rows that are not independent, classes absent from the support rows (refused, not predicted), unseen categories and support sets near the ceilings all change results in ways these metrics do not measure.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned checkpoint, validate the demonstrated tables with class coverage, condition on classification support rows, compute sample metrics against trivial and classical baselines with strict label alignment, write the input manifest and the evaluation report, score rows with class probabilities under an explicit `argmax` rule, export a DIMER-style serving bundle and rebuild an equivalent classifier from that bundle alone — without the repository being reachable. It does **not** establish benchmark superiority, domain generalisation, fairness, robustness, probability calibration, production safety, or deployment fitness.

**Next experiments:** switch `DATA_SOURCE` to `Sample: Wine` (three classes: watch one-vs-rest ROC-AUC) or `Sample: Palmer Penguins` (categorical features: watch the encoder and unseen-value counts); upload your own pre-split partitions; enable `RUN_FINE_TUNING` on a CUDA runtime and watch the holdout-based selection and the independent-test warning; raise `n_estimators` in Section 3's load expression and compare log loss and latency against LightGBM; feed the exported `outputs/tabiclv2_classifier_artifact.zip` to the companion artifact-inference notebook in a separate session.

## References

- Repository README: https://github.com/kurtvalcorza/tabicl-classifier-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/tabicl-classifier-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/tabicl-classifier-pipeline/blob/main/docs/WEIGHTS.md
- Sample dataset card: https://github.com/kurtvalcorza/tabicl-classifier-pipeline/blob/main/examples/sample-data/DATASET_CARD.md
- Upstream model: https://huggingface.co/jingang/TabICL
- Upstream library: https://github.com/soda-inria/tabicl
- TabICLv2 paper: https://arxiv.org/abs/2602.11139
- TabICL paper: https://arxiv.org/abs/2502.05564